# 01 Data Understanding

This notebook frames the business problem and reviews the two raw customer files before any modeling decisions are made.

**Business question:** Which banking customers are most likely to accept a personal loan offer?

**Project positioning:** This is a targeted marketing / propensity modeling project, not a credit approval model.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

WindowsPath('d:/Banking and Finance/Projects/canadian-bank-loan-propensity-mlops-deployment')

In [2]:
import pandas as pd

from src.config import RAW_DATA_1, RAW_DATA_2, TARGET_COL
from src.data_preprocessing import load_dataset

data1 = load_dataset(RAW_DATA_1)
data2 = load_dataset(RAW_DATA_2)

print(f"Data1 shape: {data1.shape}")
print(f"Data2 shape: {data2.shape}")

Data1 shape: (5000, 8)
Data2 shape: (5000, 7)


In [3]:
display(data1.head())
display(data2.head())

,ID,Age,CustomerSince,HighestSpend,ZipCode,HiddenScore,MonthlyAverageSpend,Level
0,1,25,1,49,91107,4,1.6,1
1,2,45,19,34,90089,3,1.5,1
2,3,39,15,11,94720,1,1.0,1
3,4,35,9,100,94112,1,2.7,2
4,5,35,8,45,91330,4,1.0,2


,ID,Mortgage,Security,FixedDepositAccount,InternetBanking,CreditCard,LoanOnCard
0,1,0,1,0,0,0,NaN
1,2,0,1,0,0,0,NaN
2,3,0,0,0,0,0,NaN
3,4,0,0,0,0,0,NaN
4,5,0,0,0,0,1,NaN


## Schema Review

The two source files are joined by `ID`. The target variable is stored in `Data2` as `LoanOnCard`.

Key review checks:

- Are customer IDs unique in each file?
- Is the target variable available and interpretable?
- Are product flags binary?
- Are tenure and spending values plausible?

In [4]:
schema_summary = pd.concat(
    {
        "Data1": pd.DataFrame({
            "feature": data1.columns,
            "dtype": data1.dtypes.astype(str).values,
            "missing_count": data1.isna().sum().values,
            "unique_values": data1.nunique(dropna=True).values,
        }),
        "Data2": pd.DataFrame({
            "feature": data2.columns,
            "dtype": data2.dtypes.astype(str).values,
            "missing_count": data2.isna().sum().values,
            "unique_values": data2.nunique(dropna=True).values,
        }),
    },
    names=["dataset", "row"],
).reset_index(level=0).reset_index(drop=True)

display(schema_summary)

,dataset,feature,dtype,missing_count,unique_values
0,Data1,ID,int64,0,5000
1,Data1,Age,int64,0,45
2,Data1,CustomerSince,int64,0,47
3,Data1,HighestSpend,int64,0,162
4,Data1,ZipCode,int64,0,467
5,Data1,HiddenScore,int64,0,4
6,Data1,MonthlyAverageSpend,float64,0,108
7,Data1,Level,int64,0,3
8,Data2,ID,int64,0,5000
9,Data2,Mortgage,int64,0,347


In [5]:
print("Duplicate IDs in Data1:", data1["ID"].duplicated().sum())
print("Duplicate IDs in Data2:", data2["ID"].duplicated().sum())

target_review = data2[TARGET_COL].value_counts(dropna=False).rename_axis(TARGET_COL).reset_index(name="count")
target_review["pct"] = target_review["count"] / len(data2)
display(target_review)

Duplicate IDs in Data1: 0
Duplicate IDs in Data2: 0


,LoanOnCard,count,pct
0,0.0,4500,0.900
1,1.0,480,0.096
2,NaN,20,0.004


## Initial Observations

The dataset is appropriate for a supervised classification problem after cleaning. Missing target records must be removed before training because they cannot contribute to supervised learning.

The next notebook performs the cleaning and preprocessing step explicitly.